# 🔗 Hungarian Algorithm for Cell Tracking — From Scratch

### An Interactive Tutorial on Bipartite Matching for Time-Lapse Microscopy

---

**Why this matters:** In cell tracking, we need to match detected cells between consecutive frames. This is a classic **assignment problem** — each cell in frame *t* should be optimally assigned to a cell in frame *t+1* based on distance and similarity.

### 📋 What You'll Learn

| Section | Topic |
|---------|-------|
| 1 | The assignment problem & why greedy matching fails |
| 2 | Hungarian algorithm — step-by-step with visualization |
| 3 | Real-world extensions: distance cutoffs, volume costs, gap closing |
| 4 | Full working tracker implementation |
| 5 | Benchmarking on synthetic cell data |

> 📚 **Prerequisites:** Basic Python, NumPy. No prior knowledge of optimization required.

---

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0e1117', 'axes.facecolor': '#0e1117',
    'axes.edgecolor': '#333', 'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0', 'xtick.color': '#aaa', 'ytick.color': '#aaa',
    'grid.color': '#222', 'font.size': 11,
})
COLORS = ['#00d2ff', '#ff6b6b', '#51cf66', '#ffd43b', '#cc5de8', '#ff922b',
          '#20c997', '#f783ac', '#748ffc', '#ced4da']

np.random.seed(42)
print('✅ Ready')

## 🎯 Section 1: The Assignment Problem

Imagine you detect **3 cells** in frame 0 and **3 cells** in frame 1. Which cell in frame 1 corresponds to which cell in frame 0?

The **cost** of assigning cell *i* to cell *j* is their Euclidean distance. We want to minimize the **total cost** of all assignments.

In [ ]:
# ── Cell 2: Toy Example — Why Greedy Fails ───────────────────────────────────────
# 3 cells in frame 0
cells_t0 = np.array([[1.0, 2.0], [3.0, 5.0], [6.0, 1.0]])
# 3 cells in frame 1 (slightly moved)
cells_t1 = np.array([[1.5, 2.5], [3.5, 4.0], [5.5, 1.5]])

# Cost matrix = pairwise distances
cost = cdist(cells_t0, cells_t1)

print('Cost Matrix (Euclidean distances):')
print('         Cell_1\'  Cell_2\'  Cell_3\'')
for i in range(3):
    print(f'Cell_{i+1}   {cost[i,0]:6.2f}   {cost[i,1]:6.2f}   {cost[i,2]:6.2f}')

# Greedy approach
print('\n--- Greedy Matching ---')
used = set()
greedy_pairs = []
greedy_cost = 0
for i in range(3):
    best_j = min([j for j in range(3) if j not in used], key=lambda j: cost[i, j])
    greedy_pairs.append((i, best_j))
    greedy_cost += cost[i, best_j]
    used.add(best_j)
    print(f'  Cell_{i+1} → Cell_{best_j+1}\' (cost: {cost[i, best_j]:.2f})')
print(f'  Total greedy cost: {greedy_cost:.2f}')

# Optimal (Hungarian) approach
print('\n--- Hungarian (Optimal) Matching ---')
row_ind, col_ind = linear_sum_assignment(cost)
hungarian_cost = 0
for r, c in zip(row_ind, col_ind):
    hungarian_cost += cost[r, c]
    print(f'  Cell_{r+1} → Cell_{c+1}\' (cost: {cost[r, c]:.2f})')
print(f'  Total optimal cost: {hungarian_cost:.2f}')

if hungarian_cost < greedy_cost:
    print(f'\n💡 Hungarian saved {greedy_cost - hungarian_cost:.2f} in total cost!')
else:
    print(f'\n✅ Both methods found the same solution (this is a simple case).')

In [ ]:
# ── Cell 3: Visual Comparison ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax_idx, (title, pairs) in enumerate([('Greedy Matching', greedy_pairs), 
                                          ('Hungarian (Optimal)', list(zip(row_ind, col_ind)))]):
    ax = axes[ax_idx]
    
    # Draw cells
    for i, (x, y) in enumerate(cells_t0):
        ax.scatter(x, y, s=200, c=COLORS[i], marker='o', zorder=5, edgecolors='white', linewidth=1.5)
        ax.annotate(f'Cell_{i+1}', (x, y), textcoords='offset points', xytext=(8, 8), 
                   fontsize=10, color=COLORS[i], fontweight='bold')
    
    for j, (x, y) in enumerate(cells_t1):
        ax.scatter(x, y, s=200, c=COLORS[j], marker='s', zorder=5, edgecolors='white', linewidth=1.5)
        ax.annotate(f'Cell_{j+1}\'', (x, y), textcoords='offset points', xytext=(8, -12),
                   fontsize=10, color=COLORS[j], fontweight='bold')
    
    # Draw links
    total = 0
    for r, c in pairs:
        ax.plot([cells_t0[r, 0], cells_t1[c, 0]], [cells_t0[r, 1], cells_t1[c, 1]],
               '-', color='white', linewidth=2, alpha=0.7, zorder=3)
        mid_x = (cells_t0[r, 0] + cells_t1[c, 0]) / 2
        mid_y = (cells_t0[r, 1] + cells_t1[c, 1]) / 2
        ax.text(mid_x, mid_y, f'{cost[r,c]:.2f}', fontsize=9, color='#ffd43b',
               ha='center', va='center', bbox=dict(boxstyle='round,pad=0.2', facecolor='#1a1a2e', alpha=0.8))
        total += cost[r, c]
    
    ax.set_title(f'{title}\nTotal Cost: {total:.2f}', fontsize=14, fontweight='bold')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.legend([mpatches.Patch(facecolor='gray', label='○ = Frame t'), 
              mpatches.Patch(facecolor='gray', label='□ = Frame t+1')],
             ['○ = Frame t', '□ = Frame t+1'], fontsize=10)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 🧮 Section 2: How the Hungarian Algorithm Works

The Hungarian algorithm (Kuhn-Munkres) solves the assignment problem in **O(n³)** time.

**Key steps:**
1. **Row reduction** — Subtract the minimum of each row from all elements in that row
2. **Column reduction** — Subtract the minimum of each column from all elements in that column
3. **Cover zeros** — Find the minimum number of lines (rows + columns) to cover all zeros
4. **If lines < n** — Modify the matrix and repeat step 3
5. **If lines = n** — An optimal assignment exists among the zeros

In practice, we use `scipy.optimize.linear_sum_assignment` which implements an efficient version.

In [ ]:
# ── Cell 4: Step-by-Step Visualization ───────────────────────────────────────────
print('=== Hungarian Algorithm Step-by-Step ===')
print(f'\nOriginal cost matrix:')
print(np.round(cost, 2))

# Step 1: Row reduction
row_reduced = cost - cost.min(axis=1, keepdims=True)
print(f'\nStep 1 — Row reduction (subtract row minimums):')
print(f'Row minimums: {cost.min(axis=1).round(2)}')
print(np.round(row_reduced, 2))

# Step 2: Column reduction
col_reduced = row_reduced - row_reduced.min(axis=0, keepdims=True)
print(f'\nStep 2 — Column reduction (subtract column minimums):')
print(f'Column minimums: {row_reduced.min(axis=0).round(2)}')
print(np.round(col_reduced, 2))

# Show zeros
zeros = np.argwhere(np.abs(col_reduced) < 1e-10)
print(f'\nZeros found at positions: {[tuple(z) for z in zeros]}')
print('These zeros indicate potential optimal assignments!')

# Final result
print(f'\n✅ Optimal assignment: {list(zip(row_ind, col_ind))}')
print(f'✅ Optimal cost: {cost[row_ind, col_ind].sum():.2f}')

## 🔬 Section 3: Real-World Extensions for Cell Tracking

The basic Hungarian algorithm assumes:
- Equal number of sources and targets
- All assignments are valid

In cell tracking, we need to handle:
1. **Distance cutoff** — Cells that are too far apart shouldn't be linked
2. **Unequal counts** — Cells can appear (birth) or disappear (death)
3. **Volume similarity** — Cells should maintain similar size
4. **Gap closing** — Cells may temporarily disappear for 1–2 frames

In [ ]:
# ── Cell 5: Full Cell Tracker Implementation ─────────────────────────────────────
@dataclass
class Cell:
    """A detected cell in one frame."""
    id: int
    frame: int
    centroid: np.ndarray          # (z, y, x) in voxels
    centroid_um: Optional[np.ndarray] = None  # (z, y, x) in µm
    volume: float = 1.0
    
    def distance_to(self, other: 'Cell', use_um: bool = True) -> float:
        a = self.centroid_um if use_um and self.centroid_um is not None else self.centroid
        b = other.centroid_um if use_um and other.centroid_um is not None else other.centroid
        return float(np.linalg.norm(a - b))


class HungarianLinker:
    """Frame-to-frame cell linker using the Hungarian algorithm.
    
    Features:
    - Distance-based cost with cutoff
    - Optional volume similarity cost
    - Confidence scoring
    """
    def __init__(self, max_distance: float = 10.0, use_volume_cost: bool = True, 
                 volume_weight: float = 0.3):
        self.max_distance = max_distance
        self.use_volume_cost = use_volume_cost
        self.volume_weight = volume_weight
    
    def link(self, cells_t: List[Cell], cells_t1: List[Cell]) -> List[Tuple[int, int, float]]:
        """Link cells between two consecutive frames.
        
        Returns: List of (source_id, target_id, confidence) tuples.
        """
        if not cells_t or not cells_t1:
            return []
        
        # Build centroid arrays
        c0 = np.array([c.centroid_um if c.centroid_um is not None else c.centroid for c in cells_t])
        c1 = np.array([c.centroid_um if c.centroid_um is not None else c.centroid for c in cells_t1])
        
        # Distance cost
        cost = cdist(c0, c1, metric='euclidean')
        
        # Optional: volume similarity cost
        if self.use_volume_cost:
            vol_t  = np.array([c.volume for c in cells_t])[:, None]
            vol_t1 = np.array([c.volume for c in cells_t1])[None, :]
            vol_ratio = np.minimum(vol_t, vol_t1) / (np.maximum(vol_t, vol_t1) + 1e-8)
            vol_cost = (1.0 - vol_ratio) * self.max_distance
            cost = (1 - self.volume_weight) * cost + self.volume_weight * vol_cost
        
        # Apply distance cutoff (set impossible links to very high cost)
        cost_masked = cost.copy()
        cost_masked[cost >= self.max_distance] = self.max_distance * 10
        
        # Solve
        row_ind, col_ind = linear_sum_assignment(cost_masked)
        
        # Filter and compute confidence
        links = []
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] < self.max_distance:
                conf = 1.0 - cost[r, c] / self.max_distance
                links.append((cells_t[r].id, cells_t1[c].id, float(conf)))
        
        return links

print('✅ HungarianLinker defined')
print(f'   max_distance: distance cutoff for valid links')
print(f'   volume_weight: blend between distance and volume cost')
print(f'   Returns: (source_id, target_id, confidence) triples')

In [ ]:
# ── Cell 6: Synthetic Cell Tracking Demo ─────────────────────────────────────────
def generate_synthetic_cells(n_frames=10, n_cells=8, drift=0.5, noise=0.3, 
                              birth_prob=0.05, death_prob=0.05):
    """Generate synthetic 2D cell trajectories with birth/death events."""
    all_cells = {}
    ground_truth = {}  # frame -> {cell_id: parent_id}
    next_id = 1
    
    # Initialize
    active = {}
    for i in range(n_cells):
        pos = np.random.uniform(0, 20, size=2)
        vol = np.random.uniform(50, 200)
        active[next_id] = {'pos': pos, 'vol': vol}
        next_id += 1
    
    for t in range(n_frames):
        cells = []
        gt = {}
        
        # Move existing cells
        new_active = {}
        for cid, info in active.items():
            if np.random.random() < death_prob and t > 0:
                continue  # Cell dies
            
            pos = info['pos'] + np.random.randn(2) * noise + np.array([drift, 0])
            vol = info['vol'] * np.random.uniform(0.95, 1.05)
            
            cell = Cell(id=cid, frame=t, centroid=pos, centroid_um=pos, volume=vol)
            cells.append(cell)
            new_active[cid] = {'pos': pos, 'vol': vol}
            gt[cid] = cid  # Same ID = ground truth link
        
        # Random birth
        if np.random.random() < birth_prob:
            pos = np.random.uniform(0, 20, size=2)
            vol = np.random.uniform(50, 200)
            cell = Cell(id=next_id, frame=t, centroid=pos, centroid_um=pos, volume=vol)
            cells.append(cell)
            new_active[next_id] = {'pos': pos, 'vol': vol}
            gt[next_id] = None  # New cell, no parent
            next_id += 1
        
        active = new_active
        all_cells[t] = cells
        ground_truth[t] = gt
    
    return all_cells, ground_truth

# Generate data
all_cells, ground_truth = generate_synthetic_cells(n_frames=10, n_cells=6)

for t in sorted(all_cells):
    print(f'Frame {t}: {len(all_cells[t])} cells — IDs: {[c.id for c in all_cells[t]]}')

In [ ]:
# ── Cell 7: Run Tracker & Visualize ──────────────────────────────────────────────
linker = HungarianLinker(max_distance=5.0, use_volume_cost=True, volume_weight=0.2)

# Track
all_links = []
sorted_frames = sorted(all_cells)
for i in range(len(sorted_frames) - 1):
    t, t1 = sorted_frames[i], sorted_frames[i + 1]
    links = linker.link(all_cells[t], all_cells[t1])
    for src, tgt, conf in links:
        all_links.append((t, src, tgt, conf))

print(f'Total links: {len(all_links)}')
print(f'Mean confidence: {np.mean([l[3] for l in all_links]):.3f}')

# Visualize trajectories
fig, ax = plt.subplots(figsize=(16, 8))

# Build trajectories from links
cell_positions = {}  # cell_id -> list of (frame, x, y)
for t in sorted_frames:
    for cell in all_cells[t]:
        if cell.id not in cell_positions:
            cell_positions[cell.id] = []
        cell_positions[cell.id].append((t, cell.centroid[0], cell.centroid[1]))

# Draw trajectories
for idx, (cid, positions) in enumerate(cell_positions.items()):
    color = COLORS[idx % len(COLORS)]
    positions = sorted(positions)
    xs = [p[1] for p in positions]
    ys = [p[2] for p in positions]
    ts = [p[0] for p in positions]
    
    ax.plot(xs, ys, '-', color=color, linewidth=2, alpha=0.7)
    ax.scatter(xs, ys, c=[color]*len(xs), s=80, zorder=5, edgecolors='white', linewidth=0.5)
    
    # Label start and end
    ax.annotate(f'ID:{cid}\nt={ts[0]}', (xs[0], ys[0]), fontsize=8, color=color,
               textcoords='offset points', xytext=(-15, 10), fontweight='bold')

# Draw links
for t, src, tgt, conf in all_links:
    src_cell = [c for c in all_cells[t] if c.id == src][0]
    t1 = t + 1
    tgt_cell = [c for c in all_cells[t1] if c.id == tgt][0]
    alpha = 0.3 + 0.5 * conf
    ax.annotate('', xy=tgt_cell.centroid[:2], xytext=src_cell.centroid[:2],
               arrowprops=dict(arrowstyle='->', color='white', alpha=alpha, lw=0.8))

ax.set_title('🔗 Cell Trajectories — Hungarian Algorithm Tracking', fontsize=16, fontweight='bold')
ax.set_xlabel('X position')
ax.set_ylabel('Y position')
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()

## 🌉 Section 4: Gap-2 Bridging

Cells can temporarily "disappear" (e.g., due to out-of-focus movement or segmentation failure). **Gap closing** attempts to reconnect cells that were unlinked for 1–2 frames.

**Strategy:** After primary linking, identify orphan cells (unlinked sources at frame *t* and unlinked targets at frame *t+2*) and attempt to link them with a reduced confidence penalty.

In [ ]:
# ── Cell 8: Gap Closing Implementation ───────────────────────────────────────────
def track_with_gap_closing(all_cells, linker, gap=2, gap_penalty=0.9):
    """Full tracking pipeline with gap-N bridging."""
    links = []
    linked_sources = {f: set() for f in all_cells}
    linked_targets = {f: set() for f in all_cells}
    sorted_frames = sorted(all_cells)
    
    # Step 1: Primary consecutive linking
    for i in range(len(sorted_frames) - 1):
        t, t1 = sorted_frames[i], sorted_frames[i + 1]
        frame_links = linker.link(all_cells[t], all_cells[t1])
        for src, tgt, conf in frame_links:
            links.append((t, src, tgt, conf))
            linked_sources[t].add(src)
            linked_targets[t1].add(tgt)
    
    primary_count = len(links)
    
    # Step 2: Gap bridging
    for g in range(2, gap + 1):
        for i in range(len(sorted_frames) - g):
            t = sorted_frames[i]
            t_gap = sorted_frames[i + g]
            
            orphan_src = [c for c in all_cells[t] if c.id not in linked_sources[t]]
            orphan_tgt = [c for c in all_cells[t_gap] if c.id not in linked_targets[t_gap]]
            
            if not orphan_src or not orphan_tgt:
                continue
            
            gap_links = linker.link(orphan_src, orphan_tgt)
            for src, tgt, conf in gap_links:
                links.append((t, src, tgt, conf * gap_penalty))
                linked_sources[t].add(src)
                linked_targets[t_gap].add(tgt)
    
    gap_count = len(links) - primary_count
    return links, primary_count, gap_count

# Run with gap closing
links, n_primary, n_gap = track_with_gap_closing(all_cells, linker, gap=2, gap_penalty=0.9)

print(f'📊 Tracking Results:')
print(f'   Primary links:     {n_primary}')
print(f'   Gap-bridged links: {n_gap}')
print(f'   Total links:       {len(links)}')
print(f'   Mean confidence:   {np.mean([l[3] for l in links]):.3f}')

## 📊 Section 5: Performance Metrics

In the BioHub competition, tracking is scored using:
- **Edge Jaccard** — overlap between predicted and ground-truth temporal edges
- **Division Jaccard** — overlap for cell division events
- **Final Score** = `edge_jaccard + 0.1 × division_jaccard`

In [ ]:
# ── Cell 9: Jaccard Metric Implementation ────────────────────────────────────────
def edge_jaccard(predicted_edges, ground_truth_edges):
    """Compute Jaccard similarity between predicted and GT edge sets."""
    pred_set = set(predicted_edges)
    gt_set = set(ground_truth_edges)
    
    if not pred_set and not gt_set:
        return 1.0
    if not pred_set or not gt_set:
        return 0.0
    
    intersection = pred_set & gt_set
    union = pred_set | gt_set
    
    return len(intersection) / len(union)

# Demo with synthetic data
pred_edges = [(l[0], l[1], l[2]) for l in links]  # (frame, src, tgt)
print(f'Predicted edges: {len(pred_edges)}')
print(f'\n🔑 Key Insight: The Hungarian algorithm guarantees the minimum-cost')
print(f'   assignment, but the overall tracking quality depends on:')
print(f'   1. Segmentation quality (are the cells detected correctly?)')
print(f'   2. Cost function design (distance + volume)')
print(f'   3. Max distance cutoff (too low = missed links, too high = false links)')
print(f'   4. Gap closing strategy (recovers temporarily lost tracks)')

In [ ]:
# ── Cell 10: Parameter Sensitivity Analysis ──────────────────────────────────────
distances = [2, 3, 5, 8, 10, 15, 20]
vol_weights = [0.0, 0.1, 0.2, 0.3, 0.5]

results = []
for d in distances:
    for vw in vol_weights:
        linker_test = HungarianLinker(max_distance=d, use_volume_cost=vw > 0, volume_weight=vw)
        test_links, n_p, n_g = track_with_gap_closing(all_cells, linker_test, gap=2)
        results.append({
            'max_dist': d, 'vol_weight': vw, 
            'n_links': len(test_links), 'n_primary': n_p, 'n_gap': n_g,
            'mean_conf': np.mean([l[3] for l in test_links]) if test_links else 0
        })

import pandas as pd
results_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Links vs max distance
for vw in [0.0, 0.2, 0.5]:
    subset = results_df[results_df['vol_weight'] == vw]
    color = COLORS[[0.0, 0.2, 0.5].index(vw)]
    axes[0].plot(subset['max_dist'], subset['n_links'], '-o', color=color,
                linewidth=2, label=f'vol_weight={vw}')
axes[0].set_title('Total Links vs Max Distance', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Max Distance (µm)')
axes[0].set_ylabel('Number of Links')
axes[0].legend()
axes[0].grid(True, alpha=0.2)

# Confidence vs max distance
for vw in [0.0, 0.2, 0.5]:
    subset = results_df[results_df['vol_weight'] == vw]
    color = COLORS[[0.0, 0.2, 0.5].index(vw)]
    axes[1].plot(subset['max_dist'], subset['mean_conf'], '-s', color=color,
                linewidth=2, label=f'vol_weight={vw}')
axes[1].set_title('Mean Link Confidence vs Max Distance', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Max Distance (µm)')
axes[1].set_ylabel('Mean Confidence')
axes[1].legend()
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print('\n📊 Full Results Table:')
display(results_df.round(3))

---

## 📌 Summary

| Concept | Key Takeaway |
|---------|--------------|
| **Assignment Problem** | Match cells between frames to minimize total cost |
| **Hungarian Algorithm** | Optimal O(n³) solver via `scipy.optimize.linear_sum_assignment` |
| **Distance Cutoff** | Prevents linking cells that are too far apart |
| **Volume Cost** | Penalizes linking cells with very different sizes |
| **Gap Closing** | Reconnects tracks interrupted for 1–2 frames |
| **Confidence** | `1 - cost/max_distance` gives a [0, 1] quality score |

### 📚 Further Reading
- Kuhn, H. W. (1955). "The Hungarian method for the assignment problem." *Naval Research Logistics*
- Jaqaman et al. (2008). "Robust single-particle tracking." *Nature Methods*
- Ulman et al. (2017). "An objective comparison of cell-tracking algorithms." *Nature Methods*

---

**If you found this tutorial helpful, please upvote! 👍**